# 5. Text to Image Search

Goals:
- Type a text query
- Embed the query
- Compute cosine similarity against 1910_embeddings.npy
- Display top-k matching images
- Show the cluster ID for each retrieved image

## Setup

In [11]:
from pathlib import Path
import json
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from sentence_transformers import SentenceTransformer
import ipywidgets as widgets
from IPython.display import display, clear_output

In [22]:
EMB_NPY = Path("1910_embeddings.npy")
EMB_TXT = Path("1910_embeddings.txt")
CLUSTERS_JSON = Path("1910_clusters.json")
PHOTOS_DIR = Path("1910_photos")

print("Using PHOTOS_DIR:", PHOTOS_DIR)

embeddings_1910 = np.load(EMB_NPY)
filenames_1910 = [ln.strip() for ln in EMB_TXT.read_text(encoding="utf-8").splitlines() if ln.strip()]
clusters = json.loads(CLUSTERS_JSON.read_text(encoding="utf-8"))

print("embeddings:", embeddings_1910.shape)
print("filenames:", len(filenames_1910))
print("clusters:", len(clusters))

if embeddings_1910.ndim != 2:
    raise ValueError(f"Expected 2D embedding matrix, got shape={embeddings_1910.shape}")
if len(filenames_1910) != embeddings_1910.shape[0]:
    raise ValueError(f"Row mismatch: {len(filenames_1910)} vs {embeddings_1910.shape[0]}")

print("Alignment check passed.")

Using PHOTOS_DIR: 1910_photos
embeddings: (72062, 512)
filenames: 72062
clusters: 2321
Alignment check passed.


In [23]:
def jpg_exists(rel_jpg: str) -> bool:
    return (PHOTOS_DIR / rel_jpg).exists()

def open_image(rel_jpg: str) -> Image.Image:
    p = PHOTOS_DIR / rel_jpg
    if not p.exists():
        raise FileNotFoundError(f"Missing image: {p}")
    return Image.open(p).convert("RGB")

def show_image_grid(rel_jpgs, title="", cols=6, figsize_per_cell=2.0, max_images=24):
    rel_jpgs = list(rel_jpgs)[:max_images]
    n = len(rel_jpgs)
    if n == 0:
        print("No images to display.")
        return

    rows = (n + cols - 1) // cols
    plt.figure(figsize=(cols * figsize_per_cell, rows * figsize_per_cell))
    for j, rel in enumerate(rel_jpgs):
        ax = plt.subplot(rows, cols, j + 1)
        try:
            img = open_image(rel)
            ax.imshow(img)
            ax.set_title(f"{j}", fontsize=8)
        except Exception:
            ax.text(0.5, 0.5, "ERR", ha="center", va="center")
        ax.axis("off")

    if title:
        plt.suptitle(title)
    plt.show()

def filename_to_reljpg(name: str) -> str:
    return f"{Path(name)}.jpg"

In [24]:
# rel_jpg -> embedding row idx
reljpg_to_idx = {filename_to_reljpg(name): i for i, name in enumerate(filenames_1910)}

# rel_jpg -> cluster_id (+ duplicate check)
jpg_to_cluster = {}
dupes = {}

for cid, jpg_list in clusters.items():
    for jpg in jpg_list:
        if jpg in jpg_to_cluster:
            dupes.setdefault(jpg, [jpg_to_cluster[jpg]]).append(cid)
        jpg_to_cluster[jpg] = cid

print("reljpg_to_idx entries:", len(reljpg_to_idx))
print("jpg_to_cluster entries:", len(jpg_to_cluster))

if dupes:
    print(f"{len(dupes)} jpgs appear in >1 cluster. Showing first 10:")
    for jpg, cids in list(dupes.items())[:10]:
        print(" ", jpg, "->", cids)
else:
    print("No duplicate images across clusters.")

reljpg_to_idx entries: 72062
jpg_to_cluster entries: 6121
No duplicate images across clusters.


In [25]:
img_vecs = embeddings_1910 / np.linalg.norm(embeddings_1910, axis=1, keepdims=True)
print("Normalized image vectors:", img_vecs.shape) # normalize for cosine similarity

Normalized image vectors: (72062, 512)


In [26]:
clustered_reljpgs = []
for cid, rels in clusters.items():
    clustered_reljpgs.extend(rels)

clustered_idxs = []
missing_in_txt = 0
for rel in clustered_reljpgs:
    i = reljpg_to_idx.get(rel)
    if i is None:
        missing_in_txt += 1
    else:
        clustered_idxs.append(i)

# Deduplicate while preserving order
seen = set()
clustered_idxs_unique = []
for i in clustered_idxs:
    if i not in seen:
        seen.add(i)
        clustered_idxs_unique.append(i)

clustered_idxs = np.array(clustered_idxs_unique, dtype=int)
img_vecs_clustered = img_vecs[clustered_idxs]

print("Total clustered rel_jpg entries:", len(clustered_reljpgs))
print("Mapped to embedding rows:", len(clustered_idxs))
print("Missing rel_jpg not found in TXT mapping:", missing_in_txt)
print("img_vecs_clustered shape:", img_vecs_clustered.shape)

Total clustered rel_jpg entries: 6121
Mapped to embedding rows: 6121
Missing rel_jpg not found in TXT mapping: 0
img_vecs_clustered shape: (6121, 512)


## Text to image search

In [28]:
print("Loading model: sentence-transformers/clip-ViT-B-32")
clip_model = SentenceTransformer("clip-ViT-B-32")


Loading model: sentence-transformers/clip-ViT-B-32


In [ ]:
def _topk_indices_from_sims(sims: np.ndarray, topk: int):
    k = min(topk, sims.shape[0])
    top = np.argpartition(-sims, k - 1)[:k]
    top = top[np.argsort(-sims[top])]
    return top

def search_text_to_image(query: str, topk: int = 12):
    """Search over ALL images."""
    query = query.strip()
    if not query:
        return []

    q = clip_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]  # (512,)
    sims = img_vecs @ q  # cosine similarity

    top_idx = _topk_indices_from_sims(sims, topk)
    results = []
    for i in top_idx:
        i = int(i)
        name = filenames_1910[i]
        rel_jpg = filename_to_reljpg(name)
        cid = jpg_to_cluster.get(rel_jpg)

        results.append({
            "index": i,
            "score": float(sims[i]),
            "filename": name,
            "rel_jpg": rel_jpg,
            "cluster_id": cid,
            "exists": bool(jpg_exists(rel_jpg)),
        })
    return results

def search_text_to_image_clustered(query: str, topk: int = 12):
    """Search only over images that appear in clusters.json."""
    query = query.strip()
    if not query:
        return []

    q = clip_model.encode([query], convert_to_numpy=True, normalize_embeddings=True)[0]  
    sims = img_vecs_clustered @ q  

    top_pos = _topk_indices_from_sims(sims, topk)
    results = []
    for pos in top_pos:
        pos = int(pos)
        global_i = int(clustered_idxs[pos])

        name = filenames_1910[global_i]
        rel_jpg = filename_to_reljpg(name)
        cid = jpg_to_cluster.get(rel_jpg)

        results.append({
            "index": global_i,
            "score": float(sims[pos]),
            "filename": name,
            "rel_jpg": rel_jpg,
            "cluster_id": cid,
            "exists": bool(jpg_exists(rel_jpg)),
        })
    return results

In [29]:
def show_search_results(query: str, topk: int = 12, cols: int = 6, clustered_only: bool = False):
    results = search_text_to_image_clustered(query, topk=topk) if clustered_only else search_text_to_image(query, topk=topk)
    if not results:
        print("No results (empty query).")
        return

    corpus = "clustered-only" if clustered_only else "all-images"
    print(f"Query: {query!r} | corpus={corpus} | topk={topk}")
    print("Top results:")
    for r in results:
        cid = r["cluster_id"] if r["cluster_id"] is not None else "None"
        ok = "OK" if r["exists"] else "MISSING"
        print(f"  idx={r['index']:>6}  score={r['score']:.4f}  cluster={cid}  {ok}  {r['rel_jpg']}")

    rels = [r["rel_jpg"] for r in results if r["exists"]]
    show_image_grid(rels, title=f"{corpus} search: {query}", cols=cols, max_images=topk)


In [30]:
query_box = widgets.Text(
    value="",
    placeholder="Type a text query, e.g., 'portrait photo', 'train station', 'newspaper advertisement'...",
    description="Query:",
    layout=widgets.Layout(width="760px")
)
topk_slider = widgets.IntSlider(value=12, min=3, max=60, step=1, description="Top-k", continuous_update=False)
cols_slider = widgets.IntSlider(value=6, min=3, max=10, step=1, description="Cols", continuous_update=False)
clustered_toggle = widgets.Checkbox(value=True, description="Search clustered only")
btn = widgets.Button(description="Search", button_style="success")
out = widgets.Output()

def on_search(_):
    with out:
        clear_output()
        show_search_results(
            query=query_box.value,
            topk=int(topk_slider.value),
            cols=int(cols_slider.value),
            clustered_only=bool(clustered_toggle.value),
        )

btn.on_click(on_search)

display(
    widgets.HBox([query_box, btn]),
    widgets.HBox([topk_slider, cols_slider, clustered_toggle]),
    out
)

Output()